In [46]:
import numpy as np
from PIL import Image
import random
from typing import Dict, Set, Tuple


def binarize_image(image_array: np.ndarray, threshold: int = 128) -> np.ndarray:
    """Преобразует изображение в бинарное (черно-белое).
    
    Args:
        image_array: Входное изображение в виде массива numpy.
        threshold: Пороговое значение для бинаризации (по умолчанию 128).
        
    Returns:
        Бинаризованный массив numpy (0 - фон, 1 - объект).
    """
    return (image_array < threshold).astype(np.uint8)


def connected_components(image_array: np.ndarray) -> np.ndarray:
    """Итеративный алгоритм."""
    height, width = image_array.shape
    labeled = np.zeros_like(image_array, dtype=np.int32)
    current_label = 1
    parent = {}  # Для хранения отношений эквивалентности
    
    def find_root(label: int) -> int:
        """Находит корневую метку с path compression."""
        while label in parent:
            parent_label = parent[label]
            if parent_label in parent:
                parent[label] = parent[parent_label]  # Path compression
            label = parent_label
        return label
    
    for y in range(height):
        for x in range(width):
            if image_array[y, x] == 0:
                continue
                
            neighbors = []
            if y > 0 and labeled[y-1, x] != 0:
                neighbors.append(labeled[y-1, x])
            if x > 0 and labeled[y, x-1] != 0:
                neighbors.append(labeled[y, x-1])
                
            if not neighbors:
                labeled[y, x] = current_label
                current_label += 1
            else:
                # Находим минимальную корневую метку среди соседей
                root_labels = [find_root(n) for n in neighbors]
                min_root = min(root_labels)
                labeled[y, x] = min_root
                
                # Объединяем все метки с минимальной корневой меткой
                for root in root_labels:
                    if root != min_root:
                        parent[root] = min_root
    
    # # Второй проход для применения path compression (опционально)
    # for y in range(height):
    #     for x in range(width):
    #         label = labeled[y, x]
    #         if label != 0:
    #             labeled[y, x] = find_root(label)
    
    return labeled


def generate_colors(num_labels: int) -> np.ndarray:
    """Генерирует случайные цвета для каждой метки."""
    colors = np.zeros((num_labels + 1, 3), dtype=np.uint8)
    for label in range(1, num_labels + 1):
        colors[label] = [random.randint(0, 255), random.randint(0, 255), random.randint(0, 255)]
    return colors


def colorize_components(labeled_array: np.ndarray) -> np.ndarray:
    """Раскрашивает компоненты случайными цветами."""
    max_label = np.max(labeled_array)
    colors = generate_colors(max_label)
    
    height, width = labeled_array.shape
    colored = np.zeros((height, width, 3), dtype=np.uint8)
    
    for y in range(height):
        for x in range(width):
            label = labeled_array[y, x]
            if label != 0:
                colored[y, x] = colors[label]
    
    return colored

In [47]:
input_path = "origins/sk.jpg" 
output_path = "results/task_5/sk_clustered.jpg"

# Загружаем изображение и преобразуем в черно-белое
img = Image.open(input_path).convert('L')
img_array = np.array(img)

binary = binarize_image(img_array)

# Находим связные компоненты
labeled = connected_components(binary)

# Раскрашиваем компоненты
colored = colorize_components(labeled)

# Сохраняем результат
result_img = Image.fromarray(colored)
result_img.save(output_path)